# Compare OOD Detection Methods

This notebook consolidates all OOD signals and compares them:

1. **Posterior variability (D_v)** — disagreement between base models
2. **Embedding distance** — Mahalanobis / k-NN in embedding space
3. **Meta-learner uncertainty** — predicted std from the trained meta-network

Each method produces a scalar score. We compare them on:
- The real observation
- A set of in-distribution test samples (baseline)

A sample is flagged OOD if its score exceeds a calibrated threshold.

In [ ]:
import sys
from pathlib import Path
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))

from sbi4atmret.config.configs import BaseConfig
from sbi4atmret.models.ModelBase import BaseModel
from sbi4atmret.models.meta_learner import (
    MetaMLP, load_base_models, meta_predict, extract_posterior_summaries,
)
from sbi4atmret.evaluation.ood_tests import (
    compute_ood_score, plot_variability,
    analyze_embeddings, plot_embedding_pca,
)

## 1. Setup

In [ ]:
config_path = project_root / "experiments/config_MiriGeminiHST_cloudfree.yaml"
with open(config_path) as f:
    config = BaseConfig(**yaml.safe_load(f))

device = "cuda" if torch.cuda.is_available() else "cpu"
n_params = len(config.prior.parameters)

# Load base models
checkpoint_paths = [
    Path("path/to/model1/states_800.pth"),
    Path("path/to/model2/states_800.pth"),
    Path("path/to/model3/states_800.pth"),
]
base_models = load_base_models(
    checkpoint_paths,
    model_builder=lambda: BaseModel(config).build(),
    device=device,
)

# Load trained meta-learner (optional)
meta_export = torch.load("path/to/meta_learner_export.pt", map_location=device)
meta_model = MetaMLP(
    input_dim=meta_export["input_dim"],
    n_params=meta_export["n_params"],
    hidden_dims=meta_export["hidden_dims"],
)
meta_model.load_state_dict(meta_export["model_state_dict"])
meta_model.to(device).eval()
include_embeddings = meta_export["include_embeddings"]

print(f"Base models: {len(base_models)}, Meta-learner loaded")

In [ ]:
# Load observation and test data
# x_obs = torch.from_numpy(observation.full_observation).unsqueeze(0).float()
# test_loader = ...
# batch_processor = ...

## 2. Method 1: Posterior Variability (D_v)

In [ ]:
dv_result = compute_ood_score(
    base_models, x_obs, n_samples=2048, method="gaussian", device=device
)
print(f"D_v = {dv_result.variability_score:.4f}")
_ = plot_variability(dv_result)
plt.show()

## 3. Method 2: Embedding Distance

In [ ]:
emb_result = analyze_embeddings(
    base_models[0], x_obs,
    dataloader=test_loader,
    batch_processor=batch_processor,
    device=device,
    max_batches=50,
)
print(f"Mahalanobis = {emb_result.mahalanobis_distance:.3f}")
print(f"k-NN dist   = {emb_result.knn_distance:.3f}")
_ = plot_embedding_pca(emb_result)
plt.show()

## 4. Method 3: Meta-Learner Uncertainty

If the meta-learner predicts large uncertainty (std), the base models
likely disagree — an indirect OOD signal.

In [ ]:
mean_obs, std_obs = meta_predict(
    base_models, meta_model, x_obs,
    n_samples=1024, include_embeddings=include_embeddings, device=device,
)

# Aggregate uncertainty: mean predicted std across parameters
meta_uncertainty = std_obs.mean().item()
print(f"Meta-learner mean uncertainty: {meta_uncertainty:.4f}")

## 5. Calibrate Thresholds on ID Data

Compute all three scores for in-distribution test samples to
establish reference distributions and thresholds.

In [ ]:
# n_cal = 30  # number of calibration samples
# dv_scores_id = []
# mahal_scores_id = []
# meta_unc_id = []
#
# for i, batches in enumerate(test_loader):
#     if i >= n_cal:
#         break
#     theta_i, x_i = batch_processor.prepare_batch(batches)
#     # Take first sample from batch
#     x_single = x_i[:1]
#
#     # D_v
#     r = compute_ood_score(base_models, x_single, n_samples=512, device=device)
#     dv_scores_id.append(r.variability_score)
#
#     # Mahalanobis (reuse test embeddings from above)
#     from sbi4atmret.evaluation.ood_tests import extract_obs_embedding, mahalanobis_distance
#     obs_emb_i = extract_obs_embedding(base_models[0], x_single, device=device).numpy()
#     mahal_scores_id.append(mahalanobis_distance(emb_result.test_embeddings, obs_emb_i))
#
#     # Meta uncertainty
#     m, s = meta_predict(base_models, meta_model, x_single,
#                         n_samples=512, include_embeddings=include_embeddings, device=device)
#     meta_unc_id.append(s.mean().item())
#
# dv_scores_id = np.array(dv_scores_id)
# mahal_scores_id = np.array(mahal_scores_id)
# meta_unc_id = np.array(meta_unc_id)

## 6. Summary Comparison

In [ ]:
# fig, axes = plt.subplots(1, 3, figsize=(14, 4))
#
# methods = [
#     ("$D_v$ (Posterior Variability)", dv_scores_id, dv_result.variability_score),
#     ("Mahalanobis Distance", mahal_scores_id, emb_result.mahalanobis_distance),
#     ("Meta-Learner Uncertainty", meta_unc_id, meta_uncertainty),
# ]
#
# for ax, (title, id_scores, obs_score) in zip(axes, methods):
#     ax.hist(id_scores, bins=15, alpha=0.7, color="steelblue", label="ID")
#     ax.axvline(obs_score, color="red", linewidth=2, linestyle="--",
#                label=f"Obs ({obs_score:.3f})")
#     percentile = (id_scores < obs_score).mean() * 100
#     ax.set_title(f"{title}\n({percentile:.0f}th percentile)")
#     ax.set_xlabel("Score")
#     ax.legend(fontsize=8)
#
# fig.suptitle("OOD Score Comparison: Observation vs In-Distribution", fontsize=13)
# plt.tight_layout()
# plt.savefig("ood_comparison.pdf", bbox_inches="tight")
# plt.show()

## 7. Decision Rule

A sample is flagged OOD if **any** of the following:
- D_v > 95th percentile of ID distribution
- Mahalanobis > 95th percentile of ID distribution
- Meta uncertainty > 95th percentile of ID distribution

Or use a combined score (e.g., average of normalized scores).

In [ ]:
# threshold_dv = np.percentile(dv_scores_id, 95)
# threshold_mahal = np.percentile(mahal_scores_id, 95)
# threshold_meta = np.percentile(meta_unc_id, 95)
#
# is_ood_dv = dv_result.variability_score > threshold_dv
# is_ood_mahal = emb_result.mahalanobis_distance > threshold_mahal
# is_ood_meta = meta_uncertainty > threshold_meta
#
# print(f"OOD by D_v:          {is_ood_dv} (score={dv_result.variability_score:.4f}, threshold={threshold_dv:.4f})")
# print(f"OOD by Mahalanobis:  {is_ood_mahal} (score={emb_result.mahalanobis_distance:.3f}, threshold={threshold_mahal:.3f})")
# print(f"OOD by Meta uncert.: {is_ood_meta} (score={meta_uncertainty:.4f}, threshold={threshold_meta:.4f})")
# print(f"\nConsensus OOD: {is_ood_dv or is_ood_mahal or is_ood_meta}")